IA Chatgpt con Arango un solo archivo

¿Qué hace este script?
Conecta a ArangoDB y crea colecciones (usuarios, prompts) y grafo (interacciones)

Usa sentence-transformers para generar embeddings de texto

Inserta esos datos en la base como documentos y vértices

Visualiza el grafo en Python con NetworkX

Realiza búsqueda semántica usando similitud de coseno

In [ ]:
# Instala dependencias si no las tienes:
# !pip install python-arango sentence-transformers numpy matplotlib networkx

In [ ]:
from arango import ArangoClient
import numpy as np
from sentence_transformers import SentenceTransformer
import networkx as nx
import matplotlib.pyplot as plt

# 1. Conexión a ArangoDB
client = ArangoClient()
db = client.db('_system', username='root', password='')

# Crear base de datos
if not db.has_database('ia_generativa'):
    db.create_database('ia_generativa')

db = client.db('ia_generativa', username='root', password='')

# 2. Crear colecciones y grafo
if not db.has_collection('prompts'):
    db.create_collection('prompts')

if not db.has_collection('usuarios'):
    db.create_collection('usuarios')

if not db.has_graph('interacciones'):
    graph = db.create_graph('interacciones')
    graph.create_vertex_collection('usuarios')
    graph.create_vertex_collection('prompts')
    graph.create_edge_definition(
        edge_collection='genero',
        from_vertex_collections=['usuarios'],
        to_vertex_collections=['prompts']
    )
else:
    graph = db.graph('interacciones')

usuarios = db.collection('usuarios')
prompts = db.collection('prompts')
genero = graph.edge_collection('genero')

# 3. Insertar un usuario
usuarios.insert({'_key': 'u1', 'nombre': 'Ana'}, overwrite=True)

# 4. Insertar prompts con embeddings reales
model = SentenceTransformer('all-MiniLM-L6-v2')

textos = [
    "Cómo usar inteligencia artificial en bases de datos",
    "Recomendaciones de libros para aprender machine learning",
    "Técnicas para optimizar consultas en grafos"
]

for i, texto in enumerate(textos):
    embedding = model.encode(texto).tolist()
    key = f"p{i+1}"
    prompts.insert({
        '_key': key,
        'texto': texto,
        'embedding': embedding
    }, overwrite=True)
    genero.insert({
        '_from': 'usuarios/u1',
        '_to': f'prompts/{key}'
    }, overwrite=True)

# 5. Visualizar el grafo con NetworkX
G = nx.DiGraph()
for u in usuarios.all():
    G.add_node(u['_key'], label=u['nombre'], color='skyblue')

for p in prompts.all():
    G.add_node(p['_key'], label=p['texto'], color='lightgreen')

for e in genero.all():
    from_node = e['_from'].split('/')[-1]
    to_node = e['_to'].split('/')[-1]
    G.add_edge(from_node, to_node)

pos = nx.spring_layout(G)
node_colors = [G.nodes[n].get('color', 'grey') for n in G.nodes]
labels = {n: G.nodes[n]['label'] for n in G.nodes}

plt.figure(figsize=(10, 6))
nx.draw(G, pos, with_labels=True, labels=labels, node_color=node_colors, node_size=2000, font_size=10)
plt.title("Grafo de Usuario y Prompts Generados")
plt.show()

# 6. Búsqueda semántica (similitud de coseno)
def cosine_similarity(vec1, vec2):
    a = np.array(vec1)
    b = np.array(vec2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

query = "Quiero aprender más sobre IA en bases de datos"
query_emb = model.encode(query).tolist()

# Comparar contra todos los embeddings en la colección
resultados = []
for p in prompts.all():
    sim = cosine_similarity(query_emb, p['embedding'])
    resultados.append((p['texto'], round(sim, 3)))

# Mostrar resultados ordenados
print("Resultados de búsqueda semántica:")
for texto, score in sorted(resultados, key=lambda x: -x[1]):
    print(f"Sim: {score} - Texto: {texto}")
